# Fine-tune vs zero-shot

Use the instructor's data to **fine-tune** a pricer, then compare it to **zero-shot** (same prompt, base model, no fine-tuning) on the same test set.

**Improvements over baseline:** (1) Cents in targets `$X.XX`, (2) filter to items with valid price & text, (3) 200 train examples + 2 epochs, (4) same prompt/target format at train and eval.

1. Load data, prepare JSONL, fine-tune (OpenAI)
2. After job succeeds: run **fine-tuned** and **zero-shot** on test; compare average error

In [ ]:
import os
import json
import re
from pathlib import Path

from dotenv import load_dotenv
from huggingface_hub import login
from openai import OpenAI

import sys
sys.path.insert(0, str(Path.cwd().resolve().parent.parent))

from pricer.items import Item
from pricer.evaluator import evaluate

load_dotenv(override=True)
login(token=os.environ["HF_TOKEN"], add_to_git_credential=True)
openai = OpenAI()

In [ ]:
DATASET = "ed-donner/items_lite"
train_raw, val_raw, test = Item.from_hub(DATASET)

def valid(item):
    text = (item.summary or item.title or "").strip()
    return item.price > 0 and len(text) > 0

train = [i for i in train_raw if valid(i)]
val = [i for i in val_raw if valid(i)]
print(f"Train: {len(train):,}  Val: {len(val):,}  Test: {len(test):,} (after dropping invalid)")

In [ ]:
def messages_for(item):
    text = (item.summary or item.title or "").strip()
    return [
        {"role": "user", "content": f"Estimate the price of this product. Respond with the price, no explanation.\n\n{text}"},
        {"role": "assistant", "content": f"${item.price:.2f}"}
    ]

def write_jsonl(items, path):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, "w") as f:
        for item in items:
            f.write(json.dumps({"messages": messages_for(item)}) + "\n")

N_TRAIN, N_VAL = 200, 50
ft_train = train[:N_TRAIN]
ft_val = val[:N_VAL]
write_jsonl(ft_train, "jsonl/fine_tune_train.jsonl")
write_jsonl(ft_val, "jsonl/fine_tune_validation.jsonl")
print(f"Wrote {N_TRAIN} train, {N_VAL} val (targets $X.XX)")

In [ ]:
with open("jsonl/fine_tune_train.jsonl", "rb") as f:
    train_file = openai.files.create(file=f, purpose="fine-tune")
with open("jsonl/fine_tune_validation.jsonl", "rb") as f:
    val_file = openai.files.create(file=f, purpose="fine-tune")

job = openai.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=val_file.id,
    model="gpt-4.1-nano-2025-04-14",
    seed=42,
    hyperparameters={"n_epochs": 2, "batch_size": 1},
    suffix="pricer-compare"
)
job_id = job.id
print(f"Job id: {job_id}")
print("Monitor at https://platform.openai.com/finetune")

In [ ]:
job = openai.fine_tuning.jobs.retrieve(job_id)
print(job.status)
ft_model_name = job.fine_tuned_model if getattr(job, "status", None) == "succeeded" else None
if ft_model_name:
    print(f"Model: {ft_model_name}")

In [ ]:
ZERO_SHOT_MODEL = "gpt-4o-mini"

assert ft_model_name, "Run the previous cell until the job status is 'succeeded'."

def test_messages(item):
    text = (item.summary or item.title or "").strip()
    return [{"role": "user", "content": f"Estimate the price of this product. Respond with the price, no explanation.\n\n{text}"}]

def predictor_fine_tuned(item):
    r = openai.chat.completions.create(model=ft_model_name, messages=test_messages(item), max_tokens=10)
    return (r.choices[0].message.content or "").strip()

def predictor_zero_shot(item):
    r = openai.chat.completions.create(model=ZERO_SHOT_MODEL, messages=test_messages(item), max_tokens=10)
    return (r.choices[0].message.content or "").strip()

In [ ]:
EVAL_SIZE = 100

print("Evaluating fine-tuned model...")
evaluate(predictor_fine_tuned, test, size=EVAL_SIZE)

print("\n---\nEvaluating zero-shot (base model)...")
evaluate(predictor_zero_shot, test, size=EVAL_SIZE)